# Lab: Vitis HLS Pragmas with Conway’s Game of Life

---

## 1. Objectives

By the end of this lab you should be able to:

* Compare a FIFO implemented in BRAM vs LUTs/flip-flops.
* Use Vitis HLS pragmas to:

  * Control storage mapping with `BIND_STORAGE`.
  * Inline functions with `INLINE`.
  * Parallelize loops with `UNROLL`.
  * Expose parallelism with `ARRAY_PARTITION`.
  * Overlap iterations with `PIPELINE`.
* Read and interpret HLS synthesis reports:

  * Latency and initiation interval (II).
  * Resource usage (LUT, FF, BRAM).
  * Estimated maximum frequency.
* Understand performance vs area trade-offs in an accelerator.

You will work with two FIFO implementations used inside a Game of Life accelerator:

* A BRAM-based FIFO (`fifo_bram`).
* A flip-flop/LUT-based FIFO (`fifo_ff`).

---

## 2. Pre-Lab Preparation

Before the lab, make sure you:

* Recall how Conway’s Game of Life works (3×3 neighbors, rules for birth/survival/death).
* Review how Vitis HLS:

  * Synthesizes C/C++ functions.
  * Reports latency and II.
  * Reports resource utilization.
* Skim the provided HLS project and locate:

  * The top function `gameoflife_compute`.
  * The two FIFO classes `fifo_bram` and `fifo_ff`.
  * The nested loops over `y` and `x`.
  * The `mat3` array for the 3×3 neighborhood.

You will see some pragmas already present but commented out in the source. In this lab, **you will add new pragma lines explicitly**, not just “uncomment” existing ones.

---

## 3. General Lab Procedure

For every change you make:

1. Modify the source code as instructed (by adding explicit `#pragma HLS ...` lines).
2. Run C synthesis in Vitis HLS.
3. Record in your lab notes:

   * Overall latency of `gameoflife_compute`.
   * II of the inner `x` loop (when pipelined).
   * LUT, FF, BRAM usage.
   * Estimated maximum frequency.

Keep a clear table of results so you can compare configurations at the end.

---

## 4. Experiment 1 – Baseline Designs

### 4.1 Setup A: BRAM FIFO Baseline

Goal: Use only the BRAM-based FIFO and measure the baseline behavior.

Actions:

1. In `gameoflife_compute`, configure the code so that **only** the BRAM-based FIFO implementation (`fifo_bram`) is used for the three line buffers.
2. Ensure that the FF-based FIFO (`fifo_ff`) is not instantiated in the design for this setup.
3. Do not add any new pragmas for this step, except those that are already in the provided baseline code.

Synthesis:

* Run synthesis.
* Record latency, II (if reported), and LUT/FF/BRAM usage.
* This is your baseline “sensible hardware designer” result.

### 4.2 Setup B: Naive FF FIFO Baseline

Goal: Use the FF-based FIFO without any helpful pragmas and observe the performance collapse.

Actions:

1. Modify `gameoflife_compute` so that:

   * The three line buffers now use the FF-based FIFO class (`fifo_ff`).
   * The BRAM-based FIFO (`fifo_bram`) is not used in this configuration.
2. Inside `fifo_ff`:

   * Keep the `data` array as declared.
   * Keep the `shift` method with the loop that shifts all entries of `data`.
   * Do **not** add any pragmas inside `fifo_ff` yet. No `BIND_STORAGE`, no `INLINE`, no `UNROLL`, no `ARRAY_PARTITION`.

Synthesis:

* Run synthesis.
* Record latency, II, resource usage.
* Compare Setup A vs Setup B and note how much slower the naive FF FIFO is.

---

## 5. Experiment 2 – Forcing fifo_ff Storage to LUTRAM

Goal: Control *where* the FIFO lives (LUTRAM instead of BRAM) without changing its algorithmic behavior.

Location:

* Inside the `fifo_ff` class, in the private section where the `data` array is declared.

Action:

1. Immediately after the line where the `data` array is declared in `fifo_ff`, **add** the following pragma line:

   Add this line directly after the declaration of the `data` array in `fifo_ff`:

   `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`

Synthesis:

* Run synthesis with this change (FF FIFO still used for the line buffers).
* Record latency, II, and resource usage.
* Compare with Experiment 1, Setup B:

  * Note any changes in BRAM vs LUT usage.
  * Note that the execution time likely did not improve significantly yet.

Concept: you changed the *implementation resource* but not the time behavior (still a big sequential shift loop).

---

## 6. Experiment 3 – Inlining fifo_ff::shift

Goal: Allow the HLS tool to see the body of `shift` inside the caller and optimize across that boundary.

Location:

* Inside `fifo_ff`, at the start of the `shift` method body.

Action:

1. Inside the `shift` method of `fifo_ff`, as the **first statement inside the function body** (right after the opening brace), **add**:

   `#pragma HLS INLINE`

Synthesis:

* Run synthesis.
* Record latency, II, and resource usage.
* Inspect the schedule viewer to verify that the `shift` logic now appears directly in the caller, not as a separate function.

The behavior is still O(max_width), but function call overhead is gone and the structure is more exposed to later optimizations.

---

## 7. Experiment 4 – Unrolling the Shift Loop in fifo_ff

Goal: Turn the sequential shift into a parallel operation across the whole FIFO.

Location:

* Inside the `shift` method of `fifo_ff`, in the for-loop that shifts elements in the `data` array.

Action:

1. Identify the for-loop in `fifo_ff::shift` that iterates over the indices of `data` to move entries (the loop that shifts elements from index `i-1` to `i`).
2. Inside that loop body, as the **first line inside the loop**, **add** the pragma:

   `#pragma HLS UNROLL`

   In other words:

   * The pragma line must be placed directly inside the loop, right after the opening brace of the loop body, before any assignment statements.

Keep from previous steps:

* The pragma `#pragma HLS INLINE` at the top of `shift`.
* The pragma `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram` after the `data` array declaration.

Synthesis:

* Run synthesis.
* Record latency, II, and resource usage.
* Compare with Experiment 3:

  * Latency should now drop dramatically.
  * LUT/FF usage should increase.
  * BRAM usage should remain low.

Here, you’ve turned the FIFO into a giant parallel shift register in one cycle.

---

## 8. Experiment 5 – Partitioning the fifo_ff Data Array

Goal: Force maximum parallel access to each element of the FIFO array.

Location:

* Inside `fifo_ff`, in the private section, near the `data` array declaration.

Action:

1. Immediately after the line that binds storage for `data`, **add** the array partition pragma:

   Place this line directly after the existing `BIND_STORAGE` pragma for `data`:

   `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`

Now, in the private section of `fifo_ff`, you should have in order:

* The declaration of `data`.
* The line `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
* The line `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`

Keep:

* The `#pragma HLS INLINE` at the start of `shift`.
* The `#pragma HLS UNROLL` inside the shift loop.

Synthesis:

* Run synthesis.
* Record latency, II, and resource usage.
* Compare with Experiment 4:

  * Check how LUT/FF usage changes.
  * Check if performance (latency/II) changes significantly.

This emphasizes how array partitioning plus unrolling can explode parallelism (and area).

---

## 9. Experiment 6 – Pipelining the Inner Game of Life Loop

Now you optimize the main computation around the FIFO.

Goal: Achieve an initiation interval close to 1 in the inner `x` loop.

Location:

* In `gameoflife_compute`, on the loop that iterates over `x` inside the double loop over `y` and `x`.

Action:

1. Identify the inner `for` loop that iterates over the `x` coordinate inside the `y` loop.
2. Directly before the `for` statement of the inner `x` loop (on the line immediately above it), **add**:

   `#pragma HLS PIPELINE II=1`

Keep:

* All previous fifo_ff pragmas:

  * `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
  * `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`
  * `#pragma HLS INLINE` in `shift`
  * `#pragma HLS UNROLL` in the shift loop

Synthesis:

* Run synthesis.
* Record:

  * The achieved II for the inner `x` loop.
  * The overall latency.
  * Resource usage and max frequency.

If the achieved II is larger than 1, the limitation may come from the neighborhood sum or other operations in the loop body.

---

## 10. Experiment 7 – Optimizing the 3×3 Neighborhood Sum (mat3)

Goal: Turn the neighbor sum into a fully combinational parallel operation and see if the pipeline improves.

Actions are in two places: array declaration and the sum loop.

### 10.1 Partition the mat3 Array

Location:

* In `gameoflife_compute`, at the declaration of the `mat3` array (the one of size 9).

Action:

1. Immediately after the line where `mat3` is declared, **add** the following pragma line:

   `#pragma HLS ARRAY_PARTITION variable=mat3 complete dim=1`

This allows parallel access to all neighbors.

### 10.2 Unroll the Neighbor Sum Loop

Location:

* In `gameoflife_compute`, in the loop that iterates over the indices of `mat3` to compute the sum of neighbors (the loop over `i` from 0 to 8, skipping the center element).

Action:

1. Inside the neighbor sum loop body, at the very beginning of the loop (right after the opening brace of the loop body), **add**:

   `#pragma HLS UNROLL`

Keep:

* The pipeline pragma on the inner `x` loop:

  `#pragma HLS PIPELINE II=1`

* All fifo_ff pragmas from previous experiments.

* The new `ARRAY_PARTITION` pragma for `mat3`.

Synthesis:

* Run synthesis.
* Record:

  * Achieved II for the `x` loop.
  * Latency.
  * Resource usage and max frequency.
* Compare with Experiment 6:

  * See if II approaches 1.
  * Note any resource usage difference.

---

## 11. Final Comparison and Analysis

Create a table with at least the following configurations:

* A: BRAM FIFO baseline (only `fifo_bram`).
* B: Naive FF FIFO (no `BIND_STORAGE`, `INLINE`, `UNROLL`, `ARRAY_PARTITION`).
* C: FF FIFO with `BIND_STORAGE` to LUTRAM.
* D: FF FIFO with `BIND_STORAGE` + `INLINE` + `UNROLL` on the shift loop.
* E: FF FIFO with `BIND_STORAGE` + `INLINE` + `UNROLL` + `ARRAY_PARTITION` on `data`.
* F: Full system with pipelined `x` loop (`#pragma HLS PIPELINE II=1` before the `x` loop).
* G: Full system with pipelined `x` loop and optimized `mat3` (`#pragma HLS ARRAY_PARTITION variable=mat3 complete dim=1` and `#pragma HLS UNROLL` in the neighbor sum loop).

For each configuration, record:

* Latency of `gameoflife_compute` (cycles).
* Achieved II of the `x` loop.
* LUT usage.
* FF usage.
* BRAM usage.
* Estimated maximum frequency.

Then, answer briefly:

1. Why is the naive FF FIFO (configuration B) so much slower than the BRAM FIFO baseline (A)?
2. Which pragma or combination of pragmas caused the biggest performance improvement for `fifo_ff`?
3. How does `#pragma HLS BIND_STORAGE variable=...` differ conceptually from `#pragma HLS ARRAY_PARTITION variable=... complete dim=...`?
4. For large grid widths, which implementation would you choose in a real design:

   * BRAM-based FIFO, or
   * Fully unrolled and partitioned FF/LUT FIFO?
     Explain your answer using your measurements.
5. Give an example scenario where you would deliberately choose:

   * A slower but smaller design, and
   * A faster but much larger design.

---

## 12. What to Hand In

Your lab report should include:

* A short introduction describing the goal of the lab.
* A summary table with all configurations (A–G) and their metrics.
* Extracts or screenshots of:

  * Performance reports (latency and II).
  * Utilization reports (LUT, FF, BRAM).
* Short answers to the analysis questions in Section 11.
* A brief reflection on what surprised you most about how much these small lines:

  * `#pragma HLS BIND_STORAGE variable=data type=ram_1p impl=lutram`
  * `#pragma HLS INLINE`
  * `#pragma HLS UNROLL`
  * `#pragma HLS ARRAY_PARTITION variable=data complete dim=1`
  * `#pragma HLS PIPELINE II=1`
  * `#pragma HLS ARRAY_PARTITION variable=mat3 complete dim=1`

  can change the hardware.

You’ve just used six lines of pragmas to turn a plodding cellular automaton into a pretty serious streaming accelerator. Not bad for a day in the lab.
